# 14장. 반복되는 분석 흐름을 안전하게 자동화하기

이 노트북은 `book/chapters/ch14_airflow_pipeline.md`와
`src/automation_pipeline.py`의 파이프라인을 단계별로 확인합니다.

핵심 원칙은 다음과 같습니다.

- Airflow를 실행하기 전에 Python 파이프라인을 검증합니다.
- 완료 주문 기준과 금액 정의를 명확히 합니다.
- 필수 컬럼, 고유 키, 파일 간 참조 관계를 검사합니다.
- 같은 입력을 다시 실행해도 중복 행이 생기지 않도록 전체 파일을 교체합니다.
- 산출물의 존재뿐 아니라 최신성, 행 수, 총합을 검증합니다.
- 비밀번호와 JWT secret은 `.env`로 분리합니다.

## 0. 실행 전 확인

원본 파일이 없다면 프로젝트 루트에서 먼저 실행합니다.

```bash
python scripts/generate_sample_data.py
```

Docker Compose 실습은 로컬 학습과 탐색을 위한 것입니다. 운영 배포용 보안·고가용성 구성이 아닙니다.

## 1. 패키지와 프로젝트 경로 설정

In [ ]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = (CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR)
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'
AIRFLOW_DIR = PROJECT_ROOT / 'automation' / 'airflow'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트 루트:', PROJECT_ROOT.resolve())
print('Airflow 폴더:', AIRFLOW_DIR.resolve())

## 2. 파이프라인 함수 불러오기

노트북과 Airflow DAG는 동일한 `src.automation_pipeline` 함수를 사용합니다.

In [ ]:
from src.automation_pipeline import (
    check_input_files, create_airflow_setup_guide, create_pipeline_task_summary,
    generate_report, generate_visualizations, run_analysis, run_local_pipeline,
    run_preprocessing, validate_outputs,
)

## 3. Task 입력·출력·재실행 규칙 확인

In [ ]:
task_summary = create_pipeline_task_summary()
task_summary

`retry_safety` 열은 같은 Task를 다시 실행했을 때 산출물이 중복되지 않도록 어떤 방식으로 저장하는지 보여 줍니다.

## 4. 원본 입력 파일 확인

In [ ]:
input_check = check_input_files(PROJECT_ROOT)
input_check

## 5. 전처리 실행

In [ ]:
preprocessing_outputs = run_preprocessing(PROJECT_ROOT)
pd.DataFrame({'dataset': preprocessing_outputs.keys(), 'path': [str(p) for p in preprocessing_outputs.values()]})

전처리 단계에서는 필수 컬럼 누락, ID 결측·중복, 타입 변환 실패, 존재하지 않는 참조, `line_total` 불일치가 있으면 즉시 중단합니다.

## 6. 완료 주문 기준 분석

In [ ]:
analysis_outputs = run_analysis(PROJECT_ROOT)
pd.DataFrame({'output': analysis_outputs.keys(), 'path': [str(p) for p in analysis_outputs.values()]})

In [ ]:
daily_sales = pd.read_csv(REPORT_DIR / 'ch14_daily_sales.csv')
category_sales = pd.read_csv(REPORT_DIR / 'ch14_category_sales.csv')
run_metadata = pd.read_csv(REPORT_DIR / 'ch14_pipeline_run_metadata.csv')
display(daily_sales.head())
display(category_sales.head())
display(run_metadata)

이 장의 금액은 `order_status == "completed"`인 주문 상세의 `quantity × unit_price` 합계입니다. 회계상 순매출이 아닙니다.

## 7. 그래프와 보고서 생성

In [ ]:
figure_outputs = generate_visualizations(PROJECT_ROOT)
report_path = generate_report(PROJECT_ROOT)
print(figure_outputs)
print(report_path)

## 8. 산출물 검증

In [ ]:
validation_log = validate_outputs(PROJECT_ROOT)
validation_log

검증은 파일 존재·최신성·행 수, 일자별·카테고리별 총합, 카테고리 비율, 보고서 범위 문구를 확인합니다.

## 9. 전체 파이프라인 한 번에 실행

In [ ]:
pipeline_result = run_local_pipeline(PROJECT_ROOT)
pipeline_result['validation_log']

터미널에서는 `python scripts/run_ch14_pipeline.py`로 같은 흐름을 실행합니다.

## 10. Docker Compose 파일 점검

In [ ]:
compose_files = [AIRFLOW_DIR / 'Dockerfile', AIRFLOW_DIR / 'docker-compose.yml', AIRFLOW_DIR / 'requirements.txt', AIRFLOW_DIR / '.env.example', AIRFLOW_DIR / 'dags' / 'ch14_local_analysis_pipeline.py']
pd.DataFrame([{'file': str(p.relative_to(PROJECT_ROOT)), 'exists': p.exists(), 'size_bytes': p.stat().st_size if p.exists() else 0} for p in compose_files])

## 11. 환경변수와 DAG 설정 확인

In [ ]:
env_example = (AIRFLOW_DIR / '.env.example').read_text(encoding='utf-8')
required_env_names = ['AIRFLOW_DB_PASSWORD', 'AIRFLOW_API_JWT_SECRET', '_AIRFLOW_WWW_USER_PASSWORD', 'AIRFLOW_DAG_TIMEZONE']
pd.DataFrame({'name': required_env_names, 'declared': [name in env_example for name in required_env_names]})

In [ ]:
dag_text = (AIRFLOW_DIR / 'dags' / 'ch14_local_analysis_pipeline.py').read_text(encoding='utf-8')
dag_checks = {
    'airflow.sdk 사용': 'from airflow.sdk import dag, task' in dag_text,
    '타임존 포함 start_date': 'pendulum.datetime' in dag_text,
    '수동 실행': 'schedule=None' in dag_text,
    'catchup 비활성화': 'catchup=False' in dag_text,
    '동시 실행 제한': 'max_active_runs=1' in dag_text,
    '실행 제한': 'execution_timeout' in dag_text,
}
pd.Series(dag_checks, name='configured')

## 12. Docker Compose 실행 순서

In [ ]:
create_airflow_setup_guide()

```bash
cd automation/airflow
cp .env.example .env
# CHANGE_ME 값 수정
docker compose build
docker compose up airflow-init
docker compose up -d
docker compose ps
```
Windows PowerShell에서는 `Copy-Item .env.example .env`를 사용합니다.

## 13. 로그인과 DAG 실행

`http://localhost:8080`에 접속하고 `.env`의 사용자 이름과 비밀번호를 입력합니다.

![Airflow 로그인 화면](../images/airflow_login_screen.svg)

로그인 후 **Dags** 메뉴에서 `ch14_local_analysis_pipeline`을 수동 실행하고 `validate_outputs`까지 성공했는지 확인합니다.

## 14. 실패 상황과 복구

원본 파일을 삭제하지 말고 이름만 임시 변경해 `check_input_files` 실패를 확인합니다. 실습 후 반드시 복구합니다. 실제 업무 데이터에서는 이런 실습을 진행하지 않습니다.

## 15. 최종 확인

In [ ]:
final_checks = pd.DataFrame({
    'check_item': [
        '모든 validation status가 ok인가?',
        '완료 주문 기준이 기록되었는가?',
        '일자별·카테고리별 총합이 일치하는가?',
        '.env.example이 자리표시자를 사용하는가?',
        'DAG가 수동 실행과 타임존을 명시하는가?',
    ],
    'status': [
        validation_log['status'].eq('ok').all(),
        run_metadata['aggregation_scope'].eq('order_status == completed').all(),
        abs(daily_sales['completed_order_amount'].sum() - category_sales['completed_order_amount'].sum()) <= 0.01,
        'CHANGE_ME' in env_example,
        dag_checks['수동 실행'] and dag_checks['타임존 포함 start_date'],
    ],
})
final_checks

## 16. 정리

Python 사전 검증, 완료 주문 범위, 멱등성, 산출물 검증, Airflow 3 TaskFlow API, 타임존, `.env` 비밀정보 분리를 함께 확인했습니다.